# Code Description
- This notebook tackles the codes about evaluation of the model for squamous cell carcinoma

## Part 0 : SCC (Kendot)

#### Code Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import torch
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision.transforms as T
from PIL import Image
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from thop import profile
import torchvision.models as models
from transformers import ViTForImageClassification
from torchsummary import summary
import timm
from timm import create_model 
import torch.nn as nn
import torch.optim as optim
import gc
from tqdm.auto import tqdm
from ptflops import get_model_complexity_info
from torch.cuda.amp import autocast, GradScaler

#### Data Loading Methods

In [ ]:
def load_image_paths(data_dir):
    image_paths, labels = [], []
    class_names = sorted([d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))])
    label_map = {name: idx for idx, name in enumerate(class_names)}
    valid_exts = ('.png', '.jpg', '.jpeg', '.bmp', '.tiff')

    for class_name in class_names:
        class_folder = os.path.join(data_dir, class_name)
        for filename in os.listdir(class_folder):
            if filename.lower().endswith(valid_exts):
                file_path = os.path.join(class_folder, filename)
                try:
                    Image.open(file_path).verify()
                    image_paths.append(file_path)
                    labels.append(label_map[class_name])
                except Exception:
                    print(f"⚠️ Skipping corrupted file: {file_path}")

    return np.array(image_paths), np.array(labels, dtype=np.int64), class_names

class CustomImageDataset(Dataset):
    def __init__(self, image_paths, labels, img_size=(224, 224), augment=False):
        self.image_paths = image_paths
        self.labels = labels
        if augment:
            self.transform = T.Compose([
                T.Resize((256, 256)),
                T.RandomCrop(img_size),
                T.RandomHorizontalFlip(),
                T.RandomVerticalFlip(),
                T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
                T.ToTensor(),
                T.Normalize(mean=[0.5]*3, std=[0.5]*3)
            ])
        else:
            self.transform = T.Compose([
                T.Resize(img_size),
                T.ToTensor(),
                T.Normalize(mean=[0.5]*3, std=[0.5]*3)
            ])

    def __len__(self): return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")
        img = self.transform(img)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return img, label


def get_dataloader(image_paths, labels, batch_size=32, augment=False, img_size=(224, 224), shuffle=True):
    ds = CustomImageDataset(image_paths, labels, img_size=img_size, augment=augment)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0, pin_memory=True)

#### Model Builder

In [ ]:
import torch.nn as nn
from torchvision import models
from timm import create_model  

def build_model(model_name, num_classes=2, pretrained=False):
    model_name = model_name.lower()
    
    if model_name == "convnext_tiny":
        model = models.convnext_tiny(pretrained=pretrained)
        in_features = model.classifier[2].in_features
        model.classifier[2] = nn.Linear(in_features, num_classes)
    
    elif model_name == "vit_base_patch16_224":
        model = models.vit_b_16(pretrained=pretrained)
        in_features = model.heads.head.in_features
        model.heads.head = nn.Linear(in_features, num_classes)
    
    elif model_name.startswith("coatnet"):
        model = create_model(model_name, pretrained=pretrained, num_classes=num_classes)
    
    else:
        raise ValueError(f"Unknown model name: {model_name}")
    
    return model


#### Model Measurements

In [ ]:
def print_model_complexity(model, device, input_size=(3, 224, 224), print_layers=True):
    model = model.to(device)
    x = torch.randn(1, *input_size).to(device)
    flops, params = profile(model, inputs=(x,), verbose=False)
    print(f"🧠 Parameters: {params/1e6:.2f} M")
    print(f"🚀 FLOPs (1x{input_size[-1]}x{input_size[-1]}): {flops/1e9:.2f} GFLOPs")
    if print_layers:
        try:
            summary(model, input_size=input_size, device=str(device))
        except Exception as e:
            print(f"(summary skipped: {e})")


def measure_latency_throughput(model, device, batch_size=32, input_size=(3,224,224), repeats=50):
    model.eval()
    dummy = torch.randn(batch_size, *input_size, device=device)

    with torch.no_grad():
        for _ in range(5): _ = model(dummy)
    if device.type == "cuda": torch.cuda.synchronize()
    start = time.time()
    with torch.no_grad():
        for _ in range(repeats): _ = model(dummy)
    if device.type == "cuda": torch.cuda.synchronize()
    total = time.time() - start
    avg_latency_ms = (total / repeats) * 1000.0
    throughput = (batch_size * repeats) / total
    print(f"⚡ Avg Latency: {avg_latency_ms:.2f} ms | 📈 Throughput: {throughput:.2f} img/s")

#### Model Training

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True)
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, total = 0.0, 0
    all_preds, all_labels = [], []

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        running_loss += loss.item() * imgs.size(0)
        preds = outputs.argmax(1)

        total += labels.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    val_loss = running_loss / total
    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    rec = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return val_loss, acc, prec, rec, f1


def plot_history(history, title, out_png):
    plt.figure(figsize=(10,4))
    plt.suptitle(title)
    plt.subplot(1,2,1)
    plt.plot(history['train_acc'], label='Train Acc')
    plt.plot(history['val_acc'], label='Val Acc')
    plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.legend(); plt.grid(True); plt.title('Accuracy')

    plt.subplot(1,2,2)
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Val Loss')
    plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.grid(True); plt.title('Loss')

    plt.tight_layout(rect=[0,0,1,0.95])
    plt.savefig(out_png, dpi=140)
    plt.show()

#### Training Codes

In [ ]:
def run_phase(model, phase_name, train_loader, val_loader, device, epochs=10, lr=1e-4, weight_decay=1e-4, out_dir="models", use_amp=True):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scaler = GradScaler() if use_amp else None
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    os.makedirs(out_dir, exist_ok=True)
    best_acc, best_state_path, best_full_path = 0.0, os.path.join(out_dir, f"{phase_name}_best.pth"), os.path.join(out_dir, f"{phase_name}_best_full.pth")

    print(f"\n▶️ Phase: {phase_name} | epochs={epochs} | lr={lr} | weight_decay={weight_decay}")
    
    for epoch in range(1, epochs+1):
        print(f"\n--- Epoch {epoch}/{epochs} ---")
        model.train()
        running_loss, running_correct, total = 0.0, 0, 0

        for i, (inputs, targets) in enumerate(train_loader):
            try:
                inputs, targets = inputs.to(device), targets.to(device)
                optimizer.zero_grad(set_to_none=True)

                if use_amp:
                    with autocast():
                        outputs = model(inputs)
                        loss = criterion(outputs, targets)
                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    outputs = model(inputs)
                    loss = criterion(outputs, targets)
                    loss.backward()
                    optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                preds = outputs.argmax(1)
                running_correct += (preds == targets).sum().item()
                total += targets.size(0)

                if i % 5 == 0:
                    print(f"Batch {i}/{len(train_loader)} | Loss: {running_loss/total:.4f} | Acc: {running_correct/total:.4f}")

            except Exception as e:
                print(f"⚠️ Skipping batch {i} due to error: {e}")

        tr_loss = running_loss / max(total, 1)
        tr_acc = running_correct / max(total, 1)

        # Validation
        va_loss, va_acc, va_prec, va_rec, va_f1 = evaluate(model, val_loader, criterion, device)

        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['val_loss'].append(va_loss)
        history['val_acc'].append(va_acc)

        print(f"Epoch {epoch:02d} | Train Loss: {tr_loss:.4f}, Acc: {tr_acc:.4f} | "
              f"Val Loss: {va_loss:.4f}, Acc: {va_acc:.4f}, P: {va_prec:.4f}, R: {va_rec:.4f}, F1: {va_f1:.4f}")

        # Save best model
        if va_acc > best_acc:
            best_acc = va_acc
            # Save state dict
            torch.save(model.state_dict(), best_state_path)
            # Save full model
            torch.save(model, best_full_path)

    return history, best_acc, best_state_path, best_full_path


def run_kfold_training(
    model_name: str,
    dataset_path: str,
    k: int = 5,
    epochs_frozen: int = 10,
    epochs_ft: int = 20,
    batch_size: int = 32,
    img_size=(224,224),
    base_lr: float = 1e-3,
    ft_lr: float = 1e-4,
    weight_decay: float = 1e-4,
    pretrained=True
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    image_paths, labels, class_names = load_image_paths(dataset_path)
    num_classes = len(class_names)

    model_dir = os.path.join("models", model_name)
    os.makedirs(model_dir, exist_ok=True)

    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    all_results = []
    fold_idx = 0

    for train_idx, val_idx in skf.split(image_paths, labels):
        fold_idx += 1
        print(f"\n==================== Fold {fold_idx}/{k} — {model_name} ====================")

        fold_dir = os.path.join(model_dir, f"fold{fold_idx}")
        os.makedirs(fold_dir, exist_ok=True)

        train_loader = get_dataloader(image_paths[train_idx], labels[train_idx],
                                      batch_size=batch_size, augment=True, img_size=img_size, shuffle=True)
        val_loader = get_dataloader(image_paths[val_idx], labels[val_idx],
                                    batch_size=batch_size, augment=False, img_size=img_size, shuffle=False)

        model = build_model(model_name, num_classes=num_classes, pretrained=pretrained).to(device)

        # Freeze backbone
        for name, param in model.named_parameters():
            param.requires_grad = False
        for name, param in model.named_parameters():
            if any(key in name for key in ['head', 'classifier', 'fc', 'bn', 'norm']):
                param.requires_grad = True

        # Phase 1: Frozen backbone
        hist_frozen, best_acc_frozen, best_state_frozen, best_full_frozen = run_phase(
            model, phase_name=f"fold{fold_idx}_frozen",
            train_loader=train_loader, val_loader=val_loader,
            device=device, epochs=epochs_frozen, lr=base_lr,
            weight_decay=weight_decay, out_dir=fold_dir
        )

        model.load_state_dict(torch.load(best_state_frozen, map_location=device))

        # Phase 2: Fine-tune all layers
        for p in model.parameters(): p.requires_grad = True
        hist_ft, best_acc_ft, best_state_ft, best_full_ft = run_phase(
            model, phase_name=f"fold{fold_idx}_finetune",
            train_loader=train_loader, val_loader=val_loader,
            device=device, epochs=epochs_ft, lr=ft_lr,
            weight_decay=weight_decay, out_dir=fold_dir
        )

        model.load_state_dict(torch.load(best_state_ft, map_location=device))
        val_loss, val_acc, val_prec, val_rec, val_f1 = evaluate(model, val_loader, nn.CrossEntropyLoss(), device)

        all_results.append({
            "fold": fold_idx,
            "best_acc_frozen": best_acc_frozen,
            "best_acc_finetune": best_acc_ft,
            "final_val_acc": val_acc,
            "final_val_precision": val_prec,
            "final_val_recall": val_rec,
            "final_val_f1": val_f1,
            "best_ckpt_frozen_state": best_state_frozen,
            "best_ckpt_frozen_full": best_full_frozen,
            "best_ckpt_finetune_state": best_state_ft,
            "best_ckpt_finetune_full": best_full_ft
        })

        # Cleanup
        del model, train_loader, val_loader
        if device.type == "cuda": torch.cuda.empty_cache()
        gc.collect()

    # Save results
    df = pd.DataFrame(all_results)
    csv_name = os.path.join(model_dir, f"{model_name}_kfold_results.csv")
    df.to_csv(csv_name, index=False)
    print(f"\n📄 Saved results: {csv_name}")

    plt.figure(figsize=(6,4))
    plt.plot(df["fold"], df["final_val_acc"], marker='o')
    plt.xlabel("Fold"); plt.ylabel("Final Val Accuracy")
    plt.title(f"{model_name} — Accuracy Across Folds (Fine-tuned best)")
    plt.grid(True)
    plt.savefig(os.path.join(model_dir, f"{model_name}_kfold_final_acc.png"), dpi=140)
    plt.show()


#### Usage

In [ ]:
dataset_path = "Dataset" 

### ConvNexT Tiny

**Testing 1**
- Folds = 5
- Epochs = 10
- Epochs (FineTuning) = 20
- Batch Size = 32
- Image Size = 224 x 224 x 3
- LR Training = 0.001
- LR Finetuning = 0.00001
- Weight Decay = 0.0001
- Type of Optimizer = AdamW

In [ ]:
model_name = "convnext_tiny" 
k = 5                
epochs_frozen = 10     
epochs_ft = 20          
batch_size = 32
img_size = (224, 224)
base_lr = 1e-3
ft_lr = 1e-4
weight_decay = 1e-4     
pretrained = True

run_kfold_training(
    model_name=model_name,
    dataset_path=dataset_path,
    k=k,
    epochs_frozen=epochs_frozen,
    epochs_ft=epochs_ft,
    batch_size=batch_size,
    img_size=img_size,
    base_lr=base_lr,
    ft_lr=ft_lr,
    weight_decay=weight_decay,
    pretrained=pretrained
)


## Part 1 : Model Evaluation Standard Metrics

## Part 2 : Model Statistical Treatment

## Part 3 : Grad-CAM usage